In [1]:
# ============================================================================
# BLOCK 1: INSTALL DEPENDENCIES
# ============================================================================

print("="*70)
print("INSTALLING DEPENDENCIES")
print("="*70)

!pip install ultralytics gradio pillow opencv-python-headless ipywidgets matplotlib -q

import torch
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io
import os
from ultralytics import YOLO

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print("="*70)

INSTALLING DEPENDENCIES



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ PyTorch: 2.6.0+cu118
✅ CUDA: True
✅ GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
# ============================================================================
# BLOCK 2: LOAD YOLOv8 HIPS & PELVIS MODEL
# ============================================================================

print("="*70)
print("LOADING YOLOv8 HIPS & PELVIS MODEL")
print("="*70)

# Path to your best YOLOv8 model (UPDATE THIS PATH)
MODEL_PATH = r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\NEW YOLO 38 Epochs\best.pt"

# Load model
model = YOLO(MODEL_PATH)

# Set to evaluation mode
model.model.eval()

print(f"✅ Model loaded: {MODEL_PATH}")
print(f"✅ Model type: YOLOv8m")
print(f"✅ Classes: {model.names}")
print("="*70)

LOADING YOLOv8 HIPS & PELVIS MODEL
✅ Model loaded: C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\NEW YOLO 38 Epochs\best.pt
✅ Model type: YOLOv8m
✅ Classes: {0: 'fracture', 1: 'no_fracture'}


In [5]:
# ============================================================================
# BLOCK 3: PREDICTION FUNCTION FOR YOLOv8 (HIPS & PELVIS)
# ============================================================================

print("="*70)
print("DEFINING PREDICTION FUNCTION")
print("="*70)

# Class names for Hips & Pelvis dataset
# Your model has: class 0 = fracture, class 1 = no_fracture
CLASS_NAMES = {0: 'fracture', 1: 'no_fracture'}

def predict_fracture(image, threshold=0.3):
    """
    Detect fractures in hip/pelvis X-ray image using YOLOv8
    Returns: (annotated_image, html_report, has_fracture, num_fractures)
    """
    # Convert to numpy array if PIL
    if isinstance(image, Image.Image):
        image = np.array(image)
    
    # Ensure RGB format
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
    
    # Run YOLOv8 prediction
    results = model(image, conf=threshold, iou=0.5, verbose=False)
    
    # Get detection info
    boxes = results[0].boxes
    num_detections = len(boxes) if boxes else 0
    
    # Filter to only show fracture detections (class 0)
    # If you want to show both fracture and no_fracture, adjust this
    fracture_indices = []
    fracture_confidences = []
    
    if num_detections > 0:
        classes = boxes.cls.cpu().numpy()
        confidences = boxes.conf.cpu().numpy()
        
        # Only keep fracture detections (class 0)
        for i, cls in enumerate(classes):
            if cls == 0:  # fracture class
                fracture_indices.append(i)
                fracture_confidences.append(confidences[i])
        
        num_fractures = len(fracture_indices)
        has_fracture = num_fractures > 0
        
        if has_fracture:
            max_confidence = float(np.max(fracture_confidences))
            avg_confidence = float(np.mean(fracture_confidences))
        else:
            max_confidence = 0.0
            avg_confidence = 0.0
    else:
        num_fractures = 0
        has_fracture = False
        max_confidence = 0.0
        avg_confidence = 0.0
    
    # Get annotated image (shows all detections)
    annotated_img = results[0].plot()
    
    # Generate HTML report
    if has_fracture:
        result_html = f"""
        <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; color: white;">
            <h1 style="margin: 0; font-size: 32px; font-weight: bold; text-align: center;">
                🚨 FRACTURE DETECTED
            </h1>
        </div>

        <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 25px; background: white; border-radius: 10px; margin-top: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 0; border-bottom: 3px solid #667eea; padding-bottom: 10px;">
                📊 Detection Details
            </h2>

            <div style="background: #f7fafc; padding: 15px; border-radius: 8px; margin: 15px 0;">
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #667eea;">•</strong> Number of fractures: <span style="font-weight: bold; color: #e53e3e;">{num_fractures}</span>
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #667eea;">•</strong> Highest confidence: <span style="font-weight: bold; color: #38a169;">{max_confidence*100:.1f}%</span>
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #667eea;">•</strong> Average confidence: <span style="font-weight: bold;">{avg_confidence*100:.1f}%</span>
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #667eea;">•</strong> Threshold used: <span style="font-weight: bold;">{threshold}</span>
                </p>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #667eea; padding-bottom: 10px;">
                🏥 Recommendation
            </h2>

            <div style="background: #fff5f5; border-left: 5px solid #e53e3e; padding: 15px; margin: 15px 0; border-radius: 5px;">
                <p style="margin: 0; font-size: 16px; font-weight: bold; color: #c53030;">
                    IMMEDIATE MEDICAL CONSULTATION RECOMMENDED
                </p>
                <p style="margin: 10px 0 0 0; font-size: 15px; color: #2d3748; line-height: 1.6;">
                    Fracture(s) detected in X-ray image.<br>
                    Please consult a qualified radiologist or orthopedic specialist.
                </p>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #667eea; padding-bottom: 10px;">
                ⚠️ Important Disclaimer
            </h2>

            <div style="background: #fffaf0; border: 2px solid #ed8936; padding: 15px; border-radius: 8px; margin-top: 15px;">
                <p style="margin: 0; font-size: 14px; color: #2d3748; line-height: 1.6;">
                    This is an AI-assisted screening tool for preliminary assessment.<br>
                    Results <strong>MUST</strong> be verified by a qualified medical professional.<br>
                    Do NOT use as sole basis for diagnosis or treatment decisions.
                </p>
            </div>
        </div>
        """
    else:
        result_html = f"""
        <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 20px; background: linear-gradient(135deg, #48bb78 0%, #38a169 100%); border-radius: 15px; color: white;">
            <h1 style="margin: 0; font-size: 32px; font-weight: bold; text-align: center;">
                ✅ NO FRACTURE DETECTED
            </h1>
        </div>

        <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 25px; background: white; border-radius: 10px; margin-top: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 0; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
                📊 Analysis Details
            </h2>

            <div style="background: #f0fff4; padding: 15px; border-radius: 8px; margin: 15px 0;">
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #48bb78;">•</strong> No fractures detected
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #48bb78;">•</strong> Threshold used: <span style="font-weight: bold;">{threshold}</span>
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #48bb78;">•</strong> Image analyzed successfully
                </p>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
                💡 Interpretation
            </h2>

            <div style="background: #e6fffa; border-left: 5px solid #38b2ac; padding: 15px; margin: 15px 0; border-radius: 5px;">
                <p style="margin: 0; font-size: 15px; color: #2d3748; line-height: 1.6;">
                    No obvious fractures detected in this X-ray image.<br>
                    However, subtle fractures may not be detected by AI.
                </p>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
                🏥 Recommendation
            </h2>

            <div style="background: #fffff0; border-left: 5px solid #ecc94b; padding: 15px; margin: 15px 0; border-radius: 5px;">
                <p style="margin: 0; font-size: 15px; color: #2d3748; line-height: 1.6;">
                    <strong>If you experience persistent pain or symptoms:</strong>
                </p>
                <ul style="margin: 10px 0; padding-left: 20px; font-size: 15px; color: #2d3748; line-height: 1.6;">
                    <li>Consult a medical professional</li>
                    <li>Clinical examination may reveal findings not visible on X-ray</li>
                    <li>Follow-up imaging may be recommended</li>
                </ul>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
                ⚠️ Important Disclaimer
            </h2>

            <div style="background: #fffaf0; border: 2px solid #ed8936; padding: 15px; border-radius: 8px; margin-top: 15px;">
                <p style="margin: 0; font-size: 14px; color: #2d3748; line-height: 1.6;">
                    This is an AI screening tool. Normal AI results do not rule out fractures.<br>
                    Always consult a qualified medical professional for proper diagnosis and treatment.
                </p>
            </div>
        </div>
        """
    
    return annotated_img, result_html, has_fracture, num_fractures

print("✅ Prediction function defined!")
print("   - Uses YOLOv8 for fast, accurate detection")
print("   - Only shows 'fracture' class detections (class 0)")
print("   - Returns annotated image and HTML report")
print("="*70)

DEFINING PREDICTION FUNCTION
✅ Prediction function defined!
   - Uses YOLOv8 for fast, accurate detection
   - Only shows 'fracture' class detections (class 0)
   - Returns annotated image and HTML report


In [9]:
# ============================================================================
# BLOCK 4: GRADIO WEB INTERFACE (FIXED - NO ZOOM)
# ============================================================================

print("="*70)
print("LAUNCHING GRADIO WEB INTERFACE (NO ZOOM)")
print("="*70)

import gradio as gr

def gradio_predict(image, threshold):
    """Wrapper for Gradio interface"""
    if image is None:
        return None, "Please upload an image."
    
    # Convert Gradio image to PIL if needed
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    
    # Run prediction
    annotated_img, result_text, has_fracture, num_fractures = predict_fracture(image, threshold)
    
    return annotated_img, result_text

# Create Gradio interface
with gr.Blocks(theme=gr.themes.Soft(), title="AETHEA Hips & Pelvis Fracture Detection") as demo:
    gr.Markdown("""
    # 🦴 AETHEA Hips & Pelvis Fracture Detection System
    
    Upload an X-ray image of a **hip or pelvis** to detect potential fractures using AI.
    
    **Model Performance:** mAP50: 79.7% | Precision: 86.9% | Recall: 75.0% | YOLOv8m
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Upload Hip/Pelvis X-ray", height=400)
            
            threshold = gr.Slider(
                minimum=0.1,
                maximum=0.9,
                value=0.3,
                step=0.05,
                label="Detection Threshold",
                info="Lower = more sensitive (more detections), Higher = more specific (fewer false positives)"
            )
            
            submit_btn = gr.Button("🔍 Detect Fracture", variant="primary", size="lg")
        
        with gr.Column(scale=1):
            # FIX: Remove height constraint to show original size
            image_output = gr.Image(label="Analysis Result", height=None)
    
    with gr.Row():
        result_output = gr.HTML(label="Diagnosis Report")
    
    submit_btn.click(
        fn=gradio_predict,
        inputs=[image_input, threshold],
        outputs=[image_output, result_output]
    )
    
    gr.Markdown("""
    ---
    ### 📋 Instructions
    1. Upload a hip or pelvis X-ray image (JPG, PNG)
    2. Adjust threshold if needed (0.3 is recommended)
    3. Click "Detect Fracture" to analyze
    
    ### 🎯 Threshold Guide
    | Threshold | Sensitivity | Use Case |
    |-----------|-------------|----------|
    | 0.2 | High | Catch subtle fractures (more false positives) |
    | 0.3 | Balanced | Default recommendation |
    | 0.5 | Low | Only confident detections |
    
    ### 📊 Model Performance
    | Metric | Value |
    |--------|-------|
    | mAP50 | 79.7% |
    | Precision | 86.9% |
    | Recall | 75.0% |
    | Inference Speed | ~0.03s/image |
    | Training Epochs | 38 |
    
    ### ⚕️ Medical Disclaimer
    This is an AI screening tool. All results must be verified by qualified healthcare professionals.
    """)

# Launch with specific settings to preserve image quality
demo.launch(share=True, debug=False, allowed_paths=["*"])

print("\n✅ Gradio interface launched! (Images shown at original size)")
print("📤 Share the public link above with others")
print("="*70)

LAUNCHING GRADIO WEB INTERFACE (NO ZOOM)
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://f80c371adc323a1d6f.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ Gradio interface launched! (Images shown at original size)
📤 Share the public link above with others


In [11]:
# ============================================================================
# IMPORTS FOR ALL ANALYSIS BLOCKS
# ============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
from tqdm import tqdm
from ultralytics import YOLO
import pandas as pd

print("✅ All imports loaded")

✅ All imports loaded


In [15]:
# ============================================================================
# CONFIDENCE THRESHOLD ANALYSIS - HIPS & PELVIS MODEL (COMPLETE)
# ============================================================================

print("="*70)
print("CONFIDENCE THRESHOLD ANALYSIS - HIPS & PELVIS")
print("="*70)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
import os

# Paths
MODEL_PATH = r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\NEW YOLO 38 Epochs\best.pt"
TEST_IMAGES_PATH = r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\test\images"
TEST_LABELS_PATH = r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\test\labels"
OUTPUT_DIR = r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the function FIRST
def compute_precision_recall_vs_threshold(model, images_dir, labels_dir, thresholds):
    """Compute precision and recall at different confidence thresholds"""
    
    # Convert to Path objects
    images_path = Path(images_dir)
    labels_path = Path(labels_dir)
    
    # Get all image files
    image_files = list(images_path.glob("*.jpg")) + list(images_path.glob("*.png"))
    
    # Store all predictions and ground truths
    all_predictions = []  # Store max confidence per image
    all_ground_truths = []  # Store whether image has fracture
    
    print(f"📊 Processing {len(image_files)} images...")
    
    for img_path in tqdm(image_files, desc="Evaluating"):
        label_path = labels_path / f"{img_path.stem}.txt"
        
        # Run inference
        results = model(str(img_path), verbose=False)
        boxes = results[0].boxes
        
        # Get max confidence if any detection
        if boxes is not None and len(boxes) > 0:
            max_conf = float(boxes.conf.cpu().numpy().max())
            all_predictions.append(max_conf)
        else:
            all_predictions.append(0.0)
        
        # Get ground truth (has fracture?)
        has_fracture = False
        if label_path.exists():
            with open(label_path, 'r') as f:
                content = f.read().strip()
                if content and len(content) > 0:
                    has_fracture = True
        all_ground_truths.append(has_fracture)
    
    # Compute metrics at each threshold
    precisions = []
    recalls = []
    f1_scores = []
    
    for thresh in thresholds:
        tp = fp = fn = tn = 0
        
        for pred_conf, has_gt in zip(all_predictions, all_ground_truths):
            pred_positive = pred_conf >= thresh
            
            if pred_positive and has_gt:
                tp += 1
            elif pred_positive and not has_gt:
                fp += 1
            elif not pred_positive and has_gt:
                fn += 1
            else:
                tn += 1
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        precisions.append(precision * 100)
        recalls.append(recall * 100)
        f1_scores.append(f1 * 100)
    
    return precisions, recalls, f1_scores, all_predictions, all_ground_truths

# Load model
model = YOLO(MODEL_PATH)

# Test thresholds
thresholds = np.arange(0.05, 0.95, 0.05)

print(f"📊 Testing across {len(thresholds)} thresholds...")

precisions, recalls, f1_scores, all_confs, all_gts = compute_precision_recall_vs_threshold(
    model, TEST_IMAGES_PATH, TEST_LABELS_PATH, thresholds
)

# Find optimal threshold (max F1 score)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]
best_precision = precisions[best_idx]
best_recall = recalls[best_idx]

print(f"\n📈 RESULTS:")
print(f"   Optimal threshold: {best_threshold:.2f}")
print(f"   F1 Score: {best_f1:.1f}%")
print(f"   Precision at optimal: {best_precision:.1f}%")
print(f"   Recall at optimal: {best_recall:.1f}%")

# Create publication-quality figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

# Plot 1: Precision/Recall vs Threshold
ax1 = axes[0]
ax1.plot(thresholds, precisions, 'b-', linewidth=2.5, label='Precision', marker='s', markersize=6)
ax1.plot(thresholds, recalls, 'g-', linewidth=2.5, label='Recall', marker='o', markersize=6)
ax1.plot(thresholds, f1_scores, 'r--', linewidth=2.5, label='F1 Score', marker='^', markersize=6)
ax1.axvline(x=best_threshold, color='purple', linestyle='--', linewidth=1.5, alpha=0.7)
ax1.scatter(best_threshold, best_f1, color='purple', s=100, zorder=5, edgecolors='black')
ax1.set_xlabel('Confidence Threshold', fontsize=12, fontweight='bold')
ax1.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax1.set_title('Hips & Pelvis: Precision/Recall vs Threshold', fontsize=14, fontweight='bold')
ax1.set_xlim(0.05, 0.95)
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.legend(loc='best', fontsize=10)
ax1.text(0.02, 0.95, f'Optimal: {best_threshold:.2f}\nF1: {best_f1:.1f}%', 
         transform=ax1.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 2: Precision-Recall Curve
ax2 = axes[1]
ax2.plot(recalls, precisions, 'b-', linewidth=2.5, marker='o', markersize=6)
ax2.scatter(best_recall, best_precision, color='red', s=100, zorder=5, edgecolors='black')
ax2.set_xlabel('Recall (%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Precision (%)', fontsize=12, fontweight='bold')
ax2.set_title('Hips & Pelvis: Precision-Recall Curve', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 100)
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.text(0.02, 0.05, f'Optimal Point\nP: {best_precision:.1f}% | R: {best_recall:.1f}%', 
         transform=ax2.transAxes, fontsize=10, verticalalignment='bottom',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'hips_pelvis_threshold_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(OUTPUT_DIR, 'hips_pelvis_threshold_analysis.png')}")

CONFIDENCE THRESHOLD ANALYSIS - HIPS & PELVIS
📊 Testing across 18 thresholds...
📊 Processing 194 images...


Evaluating: 100%|████████████████████████████████████████████████████████████████████| 194/194 [00:10<00:00, 18.41it/s]



📈 RESULTS:
   Optimal threshold: 0.05
   F1 Score: 95.7%
   Precision at optimal: 100.0%
   Recall at optimal: 91.8%


<Figure size 1400x500 with 2 Axes>

✅ Saved: C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results\hips_pelvis_threshold_analysis.png
